# Senior phenotyping pipeline — REDSEA → RESTORE → broad lineage

Reproducible, **idempotent** orchestration of the current Senior 59-plex cell-typing pipeline
(23.3M cells, 15 nPOD donors). Each compute step shells out to the canonical
`scripts/senior/*.py` (single source of truth — no logic is duplicated here) and is bookended by
cells that load the parquet outputs and render QC figures.

```
data/cells/donor_id=*/                      raw 59-marker MFI + object_id UUID
   │  scripts/senior/redsea_full.py          pixel-level lateral-spillover correction (subtract-only, 1-px band, α=1)
   ▼
data/cells_redsea/donor_id=*/               59 spillover-corrected means  (alignment r ≈ 0.998)
   │  scripts/senior/restore_normalize.py    per-image RESTORE normalization (SSC, cohort-robust)
   ▼                                         immune-ref fix: CD20/CD163←Pan_CK, CD3e←CD163
data/restore_gated_redsea/donor_id=*/       9 markers × {_pos, _norm, _log2r}
   │  scripts/senior/assign_broad_lineage.py hierarchical typing — ZERO "Unassigned"
   ▼
data/phenotype/broad/donor_id=*/            6 lineages: Epithelial / Fibroblast / Immune /
   │  scripts/senior/export_broad_class…py                Endocrine / Endothelial / Muscle
   ▼
data/phenotype/qupath_class/*.csv           UUID-keyed → QuPath (import_broad_lineage.groovy)
```

**Why this pipeline:** global clustering on this panel produced a large fake *"Unassigned"* fraction
driven by a ~22 % "pan-positive bright" population = **lateral signal bleeding** across shared cell
boundaries. REDSEA corrects the spillover at the segmentation boundary; RESTORE removes per-image
intensity drift / autofluorescence; the hierarchical rule then types **every** cell (the user
mandate: *fix the upstream cause, never invent a junk category*).

**Run it:** select the **`scvi-env`** kernel (`conda activate scvi-env`; needs `imagecodecs`,
`spams-bin`, and the vendored `external/RESTORE`). Every stage **skips if its outputs already
exist** — flip an entry in the `FORCE` dict (setup cell) to recompute. With all 15 donors already
processed, a top-to-bottom run is a fast **load-and-validate dashboard**.

In [ ]:
%matplotlib inline
import os, sys, subprocess, glob
from pathlib import Path
import pandas as pd

# --- Repo + data paths (robust whether cwd is the repo root OR notebooks/) ---
REPO = Path.cwd().resolve()
while not (REPO / "scripts" / "senior").exists() and REPO != REPO.parent:
    REPO = REPO.parent
assert (REPO / "scripts" / "senior").exists(), f"Could not locate repo root from {Path.cwd()}"
DATA = REPO / "data"
PY = sys.executable                      # use the active (scvi-env) interpreter for subprocess calls
print("Repo:", REPO)
print("Data:", DATA)
print("Python:", PY)

# --- Kernel check: the COMPUTE steps need the scvi-env interpreter (validation cells run anywhere) ---
if not (("scvi-env" in sys.prefix) or (os.environ.get("CONDA_DEFAULT_ENV") == "scvi-env")):
    print("\nWARNING: expected the 'scvi-env' conda kernel. Validation cells still run, but the "
          "REDSEA/RESTORE/lineage compute steps must run under scvi-env (conda activate scvi-env).")

# --- Force-recompute switches (default False = skip any stage whose outputs already exist) ---
FORCE = {"redsea": False, "restore": False, "lineage": False, "qupath": False}

def _has_outputs(p):
    p = Path(p)
    if not p.exists():
        return False
    if p.is_dir():                       # a hive-partitioned dir is "done" if it holds >=1 partition/file
        return (any(p.glob("donor_id=*/*.parquet")) or any(p.glob("*.parquet")) or any(p.glob("*.csv")))
    return True

def run_step(name, cmd, outputs, force=False):
    """Idempotent: skip if every output exists (unless force); else stream the subprocess live."""
    outs = [Path(o) for o in (outputs if isinstance(outputs, (list, tuple)) else [outputs])]
    if all(_has_outputs(o) for o in outs) and not force:
        print(f"[SKIP] {name}: outputs present -> " + ", ".join(str(o.relative_to(REPO)) for o in outs))
        return
    cmd = [str(c) for c in cmd]
    print(f"[RUN ] {name}\n       {' '.join(cmd)}\n" + "-" * 72)
    proc = subprocess.Popen(cmd, cwd=str(REPO), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    print("-" * 72)
    if proc.returncode != 0:
        raise RuntimeError(f"{name} FAILED (exit {proc.returncode})")
    print(f"[DONE] {name}")

# --- Donor discovery + disease-status lookup (used by the validation cells) ---
donors = sorted(p.name.split("=", 1)[1] for p in (DATA / "cells").glob("donor_id=*"))
print(f"\n{len(donors)} donors discovered:", ", ".join(donors))

def find_donor_meta():
    for c in (DATA / "donor_metadata_panc.xlsx", REPO.parent / "donor_metadata_panc.xlsx",
              REPO / "donor_metadata_panc.xlsx"):
        if Path(c).exists():
            return Path(c)
    return None

def donor_status_map():
    mp = find_donor_meta()
    if not mp:
        return {}
    meta = pd.read_excel(mp)
    dcol = next((c for c in meta.columns if c.lower() in ("donor_id", "donor", "case_id")), meta.columns[0])
    scol = next((c for c in meta.columns if "status" in c.lower()), None)
    return {} if scol is None else {str(k): str(v) for k, v in zip(meta[dcol], meta[scol])}

_STATUS_ORDER = {"ND": 0, "AAB": 1, "Aab+": 1, "AAb+": 1, "T1D": 2}
def status_sort(values):
    return sorted(set(values), key=lambda s: _STATUS_ORDER.get(str(s), 9))

## Step A — per-cell geometry export (prerequisite, outside the kernel)

REDSEA needs the segmentation boundary of every cell. These are exported once from QuPath
(headless Groovy, `~/IO60panc2nd/scripts/export_cells_geojson.groovy`) — **not** runnable from this
kernel because it needs QuPath + the multi-hundred-GB `.qptiff` images. Each GeoJSON feature `id` is
the detection **UUID** (== our `object_id`); coords are full-resolution pixels.

```bash
# Close the QuPath GUI first (it holds ~400 GB RAM). One --image per invocation:
while IFS= read -r img; do
  ~/QuPath/bin/QuPath script --project ~/IO60panc2nd/project.qpproj --image "$img" \
     ~/IO60panc2nd/scripts/export_cells_geojson.groovy
done < donor_images.txt          # -> data/redsea_scratch/geojson/cells__<image>.geojson
```
The next cell only *verifies* the export (it is not needed when `data/cells_redsea/` already exists).

In [ ]:
geo = DATA / "redsea_scratch" / "geojson"
n_geo = len(list(geo.glob("cells__*.geojson"))) if geo.exists() else 0
print(f"GeoJSON exports: {n_geo}" + (f"   ({geo})" if n_geo else "   -- MISSING"))
if n_geo == 0:
    print("NOTE: REDSEA (Step B) needs these. If data/cells_redsea already exists, Step B is skipped\n"
          "      and the GeoJSONs are not required for a load-and-validate run.")

## Step B — REDSEA pixel-level spillover correction

`redsea_full.py` rasterizes each donor's GeoJSON to an int32 instance mask, reads the `.qptiff`
one channel at a time, builds an 8-connected (gap-bridged) contact matrix, and compensates
`corrected_mean = clip(data − α·(F @ edge), 0) / cell_area_px`. Two validated, non-obvious defaults:
**subtract-only** (`--comp-mode 0` — reinforce is a no-op in confluent tissue) and a **thin 1-px
band** (`--edge-radius 0`) with **α = 1**. ~8 min/donor × 15. Writes `data/cells_redsea/`.

In [ ]:
run_step("REDSEA",
         [PY, "scripts/senior/redsea_full.py", "--all"],
         outputs=[DATA / "cells_redsea"],
         force=FORCE["redsea"])

In [ ]:
import pyarrow.dataset as ds
from IPython.display import Image, display

# Alignment sanity: REDSEA-corrected means reproduce the raw QuPath means at r ~ 0.998
# (redsea_full.py logs this per donor). Spot-check one donor + one abundant channel.
d0 = donors[0]
raw_cols = set(ds.dataset(DATA / "cells" / f"donor_id={d0}", format="parquet").schema.names)
cor_cols = set(ds.dataset(DATA / "cells_redsea" / f"donor_id={d0}", format="parquet").schema.names)
shared = (raw_cols & cor_cols) - {"object_id", "donor_id", "image", "cell_area", "cell_area_px",
                                  "cell_region", "islet_num"}
ch = "Pan_Cytokeratin" if "Pan_Cytokeratin" in shared else sorted(shared)[0]

raw = ds.dataset(DATA / "cells" / f"donor_id={d0}", format="parquet").to_table(
        columns=["object_id", ch]).to_pandas()
cor = ds.dataset(DATA / "cells_redsea" / f"donor_id={d0}", format="parquet").to_table(
        columns=["object_id", ch]).to_pandas()
m = raw.merge(cor, on="object_id", suffixes=("_raw", "_redsea"))
r = m[f"{ch}_raw"].corr(m[f"{ch}_redsea"])
print(f"donor {d0}: {ch}  raw vs REDSEA  r = {r:.4f}  (n = {len(m):,})")
print(f"           mean {m[f'{ch}_raw'].mean():.1f} -> {m[f'{ch}_redsea'].mean():.1f}  "
      f"(spillover is subtracted, so the corrected mean is <= raw)")

# Committed validation figure: impossible cross-compartment co-positives collapse ~0.46% -> 0.05%.
fig = DATA / "redsea_full_validation.png"
if fig.exists():
    display(Image(filename=str(fig)))
else:
    print("(data/redsea_full_validation.png not found — run redsea_full_validate.py to regenerate)")

## Step C — RESTORE per-image normalization (on the REDSEA-corrected cohort)

`restore_normalize.py` fits the RESTORE 2-cluster **SSC** model per image on each mutually-exclusive
marker pair, takes the reference-positive cluster as the target's negative population, and sets the
threshold at `mean + 3σ`; degenerate per-image thresholds are imputed with the cohort median
(`--robust`, on by default). The **immune negative-control fix** is the key deviation from the paper:
`CD20/CD163 ← Pan_Cytokeratin` (their background scales with cell brightness) but `CD3e ← CD163`
(keep immune-vs-immune — Pan_CK would destroy the validated T-cell infiltration signal).

Because we run on the **REDSEA-corrected** cells, the dirs are passed explicitly (the defaults point
at raw `data/cells`): `--cells-dir data/cells_redsea --out-dir data/restore_redsea
--gated-dir data/restore_gated_redsea`. Writes `data/restore_gated_redsea/` ({m}_pos / _norm /
_log2r) — the broad-lineage input.

In [ ]:
run_step("RESTORE (REDSEA-corrected cohort)",
         [PY, "scripts/senior/restore_normalize.py",
          "--cells-dir",  "data/cells_redsea",
          "--out-dir",    "data/restore_redsea",
          "--gated-dir",  "data/restore_gated_redsea",
          "--thresh-csv", "data/restore_thresholds_redsea.csv"],
         outputs=[DATA / "restore_gated_redsea"],
         force=FORCE["restore"])

In [ ]:
# Gate the 3 extra lineage markers (B3TUBB->Neural, CD99->Endocrine, MPO->Neutrophil) into a SEPARATE
# dir so the validated 10-marker gates in restore_gated_redsea stay byte-identical. Merged by object_id
# in assign_broad_lineage.py / plot_celltype_markers.py.
run_step("RESTORE extra (B3TUBB/CD99/MPO)",
         [PY, "scripts/senior/restore_normalize.py",
          "--marker-pairs", "B3TUBB:Pan_Cytokeratin,CD99:Pan_Cytokeratin,MPO:Pan_Cytokeratin",
          "--cells-dir",  "data/cells_redsea",
          "--out-dir",    "data/restore_redsea_extra",
          "--gated-dir",  "data/restore_gated_redsea_extra",
          "--thresh-csv", "data/restore_thresholds_extra.csv"],
         outputs=[DATA / "restore_gated_redsea_extra"],
         force=FORCE["restore"])


In [ ]:
# (a) Per-image positive fractions written by RESTORE -> mean per marker across the 15 images.
pf = next((p for p in (DATA / "restore_redsea" / "positive_fractions.csv",
                       DATA / "restore" / "positive_fractions.csv") if p.exists()), None)
if pf is not None:
    frac = pd.read_csv(pf)
    mcol = "marker" if "marker" in frac.columns else frac.columns[1]
    fcol = "frac_pos" if "frac_pos" in frac.columns else next(c for c in frac.columns if "frac" in c.lower())
    print(f"RESTORE positive fractions  ({pf.relative_to(REPO)}):")
    display((100 * frac.groupby(mcol)[fcol].mean()).round(2).sort_values(ascending=False)
            .to_frame("mean % positive across images"))
else:
    print("positive_fractions.csv not found (run Step C).")

# (b) Biological hallmark recovery: beta-cell (INS+) loss + T-cell (CD3e+) gain ND -> Aab+ -> T1D.
status = donor_status_map()
if status:
    rows = []
    for d in donors:
        t = ds.dataset(DATA / "restore_gated_redsea" / f"donor_id={d}", format="parquet").to_table(
                columns=["INS_pos", "CD3e_pos"]).to_pandas()
        rows.append({"status": status.get(d, "?"),
                     "INS+ %": 100 * t["INS_pos"].mean(), "CD3e+ %": 100 * t["CD3e_pos"].mean()})
    trend = pd.DataFrame(rows).groupby("status")[["INS+ %", "CD3e+ %"]].mean().round(2)
    print("\nHallmark recovery (mean % positive by disease status):")
    display(trend.reindex(status_sort(trend.index)))
else:
    print("\n(donor_metadata_panc.xlsx not found — skipping disease-trend recompute; see Step D figure.)")

## Step D — hierarchical broad-lineage assignment (8 classes, zero "Unassigned")

`assign_broad_lineage.py` types every cell with a **hierarchical** rule (specific `_pos` lineages
override the structural background). Precedence high→low:

1. **Endocrine** if `INS_pos | GCG_pos | SST_pos | CD99_bright`  (β / α / δ hormones; **bright CD99**
   = `CD99_norm ≥ 3` marks the "other" endocrine types we lack markers for — PP/epsilon/EC. The plain
   `mean+3σ` CD99 gate is broad and over-calls a non-islet population; the ≥3× bright cut is validated
   islet-coherent, ~36% islet-core like INS/GCG/SST.)
2. **Immune** elif `CD3e_pos | CD20_pos | CD163_pos`  (T / B / macrophage)
3. **Endothelial** elif `CD31_pos`
4. **Neutrophil** elif `MPO_pos`
5. **Neural** elif `B3TUBB_pos`  (nerves / Schwann — else mis-typed Fibroblast)
6. else **structural** = argmax of (`Pan_Cytokeratin`, `Vimentin`, `SMA`) `_norm` →
   **Epithelial / Fibroblast / Muscle**, with cells below *all* structural thresholds defaulted to
   **Epithelial** (`epi_default`).

B3TUBB/CD99/MPO come from the separate `restore_gated_redsea_extra` gate (the RESTORE-extra cell above).
Not a plain argmax: RESTORE's `mean+3σ` flags only the brightest ~20 % of an abundant marker, so the dim
acinar majority is sub-threshold and a naive argmax would scatter it to low-threshold markers. Writes
`data/phenotype/broad/`.

In [ ]:
run_step("Broad-lineage assignment",
         [PY, "scripts/senior/assign_broad_lineage.py"],
         outputs=[DATA / "phenotype" / "broad"],
         force=FORCE["lineage"])

In [ ]:
from collections import Counter
from IPython.display import Image, display
import pyarrow.dataset as ds

# Read per donor (donor_id comes from the loop, not the file) — avoids the hive int32-vs-string
# donor_id conflict and keeps memory low.
_LIN = ["Epithelial", "Fibroblast", "Muscle", "Endothelial", "Endocrine", "Immune"]
status = donor_status_map()
total, epi_def = 0, 0
lin_counts, status_lin = Counter(), {}
for d in donors:
    t = ds.dataset(DATA / "phenotype" / "broad" / f"donor_id={d}", format="parquet").to_table(
            columns=["broad_lineage", "epi_default"]).to_pandas()
    total += len(t); epi_def += int(t["epi_default"].sum())
    vc = t["broad_lineage"].value_counts().to_dict()
    lin_counts.update(vc)
    status_lin.setdefault(status.get(d, "?"), Counter()).update(vc)

assert "Unassigned" not in lin_counts, "FAIL: found 'Unassigned' cells — the mandate is zero!"
print(f"Total cells typed: {total:,}\nZERO 'Unassigned'  -- OK\n")

vc = pd.Series(lin_counts).sort_values(ascending=False)
print("Lineage counts:")
display(vc.to_frame("cells").assign(**{"%": (100 * vc / total).round(2)}))
print(f"epi_default (sub-threshold structural -> Epithelial): {100 * epi_def / total:.1f}% of cells\n")

# Composition (% of cells) by disease status
if status:
    comp = pd.DataFrame(status_lin).T.fillna(0)
    comp = comp.div(comp.sum(axis=1), axis=0).mul(100).round(1)
    comp = comp.reindex(index=status_sort(comp.index),
                        columns=[c for c in _LIN if c in comp.columns])
    print("Composition (% of cells) by disease status:")
    display(comp)

# Committed 3-panel figure (per-donor stacked / by-status means / disease trend).
fig = DATA / "phenotype" / "broad_lineage_composition.png"
if fig.exists():
    display(Image(filename=str(fig)))

## Step E — export for QuPath visual inspection (optional)

`export_broad_class_for_qupath.py` writes one tiny UUID-keyed CSV per donor
(`object_id, broad_lineage, image`). In QuPath 0.7, run `scripts/groovy/import_broad_lineage.groovy`
(matches detections by `PathObject.getID()` == `object_id` — exact, no centroid rounding — and sets
PathClass + lineage color), then `scripts/groovy/view phenotypes.groovy` to toggle classes on/off.

In [ ]:
run_step("QuPath export",
         [PY, "scripts/senior/export_broad_class_for_qupath.py"],
         outputs=[DATA / "phenotype" / "qupath_class"],
         force=FORCE["qupath"])

## Pipeline status

In [ ]:
stages = [
    ("Raw cells",         DATA / "cells",                    "donor_id=*"),
    ("REDSEA corrected",  DATA / "cells_redsea",             "donor_id=*"),
    ("RESTORE gated",     DATA / "restore_gated_redsea",     "donor_id=*"),
    ("Broad lineage",     DATA / "phenotype" / "broad",      "donor_id=*"),
    ("QuPath class CSVs", DATA / "phenotype" / "qupath_class","*.csv"),
]
rows = []
for name, p, patt in stages:
    n = len(list(p.glob(patt))) if p.exists() else 0
    rows.append({"stage": name, "path": str(p.relative_to(REPO)),
                 "partitions/files": n, "status": "OK" if n else "MISSING"})
display(pd.DataFrame(rows))

## Step F — cell type × marker profile (heatmap + dotplot)

The final validation: does each broad lineage actually express its defining markers? Two
complementary views, computed as **exact** per-lineage means over all 23.3M typed cells (one
streaming pass per donor — no subsampling):

- **Dotplot** — lineage × the 10 RESTORE lineage markers. Dot **size** = % of cells positive
  (`<m>_pos`); dot **color** = mean RESTORE-normalized intensity (`<m>_norm`), **scaled per marker**
  so each lineage's top marker is visible (Epithelial's Pan_CK is sub-threshold for the dim acinar
  majority, so on an absolute scale it would render as an invisible dark dot). Each lineage lights up
  its own marker (Epithelial→Pan_CK, Fibroblast→Vimentin, Muscle→SMA, Endothelial→CD31,
  Endocrine→INS/GCG/**SST**, Immune→CD3e/CD20/CD163).
- **Heatmap** — lineage × the full REDSEA-corrected marker panel, mean expression z-scored per
  marker across lineages, for the broader phenotypic signature.

In [ ]:
from IPython.display import Image, display

# Cell-type × marker dotplot + heatmap — generated by the canonical script (single source of truth;
# this is a thin orchestrator like the compute steps above, not duplicated plotting logic). The
# streaming per-lineage aggregation + NaN-safe means now live in scripts/senior/plot_celltype_markers.py.
run_step("celltype × marker figures",
         [PY, "scripts/senior/plot_celltype_markers.py"],
         outputs=[DATA / "phenotype" / "celltype_marker_dotplot.png",
                  DATA / "phenotype" / "celltype_marker_heatmap.png"],
         force=FORCE.get("lineage", False))

for fig in (DATA / "phenotype" / "celltype_marker_dotplot.png",
            DATA / "phenotype" / "celltype_marker_heatmap.png"):
    if fig.exists():
        display(Image(filename=str(fig)))
    else:
        print(f"({fig.name} not found — run plot_celltype_markers.py)")
